# COMP560 Qwen3-VL + Tinker Colab Notebook

This notebook is a Colab-ready starter for testing Tinker with `Qwen/Qwen3-VL-30B-A3B-Instruct` on the same face-verification task as the MobileFaceNet pipeline.

It uses `models/qwen_vl_tinker_finetune.py` to:
- preview pairwise multimodal training examples
- export a JSONL SFT dataset
- launch a first Tinker fine-tuning run
- optionally sample from a saved Tinker checkpoint through the OpenAI-compatible endpoint

In [ ]:
import os
import subprocess
from pathlib import Path

repo_root = Path("/content/560-FaceRecognition")
if not repo_root.exists():
    subprocess.run([
        "git",
        "clone",
        "https://github.com/ethan-yountz/560-FaceRecognition",
        str(repo_root),
    ], check=True)
else:
    print(f"Using existing repo at {repo_root}")

project_root = repo_root / "project-fr"
os.chdir(project_root)
print(Path.cwd())

In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess

drive.mount("/content/drive", force_remount=True)

drive_root = Path("/content/drive/MyDrive")
comp560_dir = drive_root / "Comp560"
candidate_paths = [
    comp560_dir / "datasets.tar.gz",
    drive_root / "datasets.tar.gz",
]
datasets_tar = next((path for path in candidate_paths if path.exists()), None)

if datasets_tar is None:
    if comp560_dir.exists():
        visible_files = sorted(path.name for path in comp560_dir.iterdir())[:20]
        raise FileNotFoundError(
            f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Files in {comp560_dir}: {visible_files}"
        )
    raise FileNotFoundError(
        f"Could not find datasets.tar.gz. Checked: {candidate_paths}. Folder not found: {comp560_dir}"
    )

subprocess.run(["tar", "-xf", str(datasets_tar)], check=True)
print(f"Extracted {datasets_tar} into {Path.cwd()}")

In [ ]:
from pathlib import Path
import subprocess

dataset_dir = Path("datasets/dataset_a")
if not dataset_dir.exists():
    available_datasets = sorted(path.name for path in Path("datasets").iterdir() if path.is_dir())
    raise FileNotFoundError(
        f"Missing dataset_a after extraction. Available dataset directories: {available_datasets}"
    )

images_dir = dataset_dir / "images"
images_tar = dataset_dir / "images.tar.gz"

if images_dir.exists():
    print(f"Using existing extracted images at {images_dir}")
elif images_tar.exists():
    subprocess.run(["tar", "-xf", images_tar.name], cwd=images_tar.parent, check=True)
    print(f"Extracted {images_tar}")
else:
    raise FileNotFoundError(f"Missing both extracted images dir and image archive under {dataset_dir}")

In [ ]:
!pwd
!ls
!ls models
!ls notebooks

## Install Tinker Dependencies

This uses regular `pip` inside Colab. If you need a newer Tinker Cookbook build later, replace the stable install with the nightly GitHub install.

In [ ]:
%pip install -q tinker tinker-cookbook marimo chz openai pandas pyarrow

In [ ]:
import tinker
import tinker_cookbook

print("tinker import ok")
print("tinker_cookbook import ok")

## Tinker API Key

Paste your Tinker API key into the blank string below.

In [ ]:
import os

TINKER_API_KEY = ""

if not TINKER_API_KEY.strip():
    raise ValueError("Set TINKER_API_KEY in this cell before continuing.")

os.environ["TINKER_API_KEY"] = TINKER_API_KEY.strip()
print("TINKER_API_KEY loaded into the environment.")

## Run Configuration

Adjust these values before generating the JSONL dataset or launching training.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("datasets/dataset_a")
TRAIN_METADATA = None
TRAIN_PAIRS = None
OUTPUT_JSONL = DATA_ROOT / "qwen_face_train.jsonl"
MODEL_NAME = "Qwen/Qwen3-VL-30B-A3B-Instruct"
MAX_EXAMPLES = 5000
IMAGES_PER_TEMPLATE = 2
INLINE_IMAGES = False

LOG_PATH = "./logs/qwen_face_tinker"
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
EVAL_EVERY = 50
LORA_RANK = 32
MAX_LENGTH = 32768
SEED = 42

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Missing dataset root: {DATA_ROOT}. Make sure dataset_a is available in this Colab runtime."
    )

print(f"DATA_ROOT={DATA_ROOT}")
print(f"OUTPUT_JSONL={OUTPUT_JSONL}")
print(f"MODEL_NAME={MODEL_NAME}")

## Preview Generated Training Examples

This shows how the face-verification task is converted into Qwen3-VL conversations.

In [ ]:
preview_cmd = [
    "python",
    "models/qwen_vl_tinker_finetune.py",
    "preview",
    "--data_root",
    str(DATA_ROOT),
    "--output_jsonl",
    str(OUTPUT_JSONL),
    "--max_examples",
    str(MAX_EXAMPLES),
    "--images_per_template",
    str(IMAGES_PER_TEMPLATE),
    "--seed",
    str(SEED),
    "--model_name",
    MODEL_NAME,
    "--num_preview",
    "2",
]

if TRAIN_METADATA:
    preview_cmd.extend(["--train_metadata", TRAIN_METADATA])
if TRAIN_PAIRS:
    preview_cmd.extend(["--train_pairs", TRAIN_PAIRS])
if INLINE_IMAGES:
    preview_cmd.append("--inline_images")

print(" ".join(preview_cmd))
!{" ".join(preview_cmd)}

## Export JSONL Dataset

In [ ]:
prepare_cmd = [
    "python",
    "models/qwen_vl_tinker_finetune.py",
    "prepare",
    "--data_root",
    str(DATA_ROOT),
    "--output_jsonl",
    str(OUTPUT_JSONL),
    "--max_examples",
    str(MAX_EXAMPLES),
    "--images_per_template",
    str(IMAGES_PER_TEMPLATE),
    "--seed",
    str(SEED),
    "--model_name",
    MODEL_NAME,
]

if TRAIN_METADATA:
    prepare_cmd.extend(["--train_metadata", TRAIN_METADATA])
if TRAIN_PAIRS:
    prepare_cmd.extend(["--train_pairs", TRAIN_PAIRS])
if INLINE_IMAGES:
    prepare_cmd.append("--inline_images")

print(" ".join(prepare_cmd))
!{" ".join(prepare_cmd)}

## Launch a First Tinker Run

This uses the prepared JSONL file and starts a Tinker Cookbook supervised fine-tuning job.

In [ ]:
train_cmd = [
    "python",
    "models/qwen_vl_tinker_finetune.py",
    "train",
    "--data_root",
    str(DATA_ROOT),
    "--output_jsonl",
    str(OUTPUT_JSONL),
    "--max_examples",
    str(MAX_EXAMPLES),
    "--images_per_template",
    str(IMAGES_PER_TEMPLATE),
    "--seed",
    str(SEED),
    "--model_name",
    MODEL_NAME,
    "--log_path",
    LOG_PATH,
    "--batch_size",
    str(BATCH_SIZE),
    "--learning_rate",
    str(LEARNING_RATE),
    "--num_epochs",
    str(NUM_EPOCHS),
    "--eval_every",
    str(EVAL_EVERY),
    "--lora_rank",
    str(LORA_RANK),
    "--max_length",
    str(MAX_LENGTH),
]

if TRAIN_METADATA:
    train_cmd.extend(["--train_metadata", TRAIN_METADATA])
if TRAIN_PAIRS:
    train_cmd.extend(["--train_pairs", TRAIN_PAIRS])
if INLINE_IMAGES:
    train_cmd.append("--inline_images")

print(" ".join(train_cmd))

In [ ]:
!{" ".join(train_cmd)}

## Optional: Sample From a Saved Tinker Checkpoint

After training, set `MODEL_PATH` to a Tinker sampler checkpoint path such as:

`tinker://.../sampler_weights/000080`

Leave this blank until you have a real checkpoint.

In [ ]:
from openai import OpenAI

BASE_URL = "https://tinker.thinkingmachines.dev/services/tinker-prod/oai/api/v1"
MODEL_PATH = ""

if not MODEL_PATH.strip():
    raise ValueError("Set MODEL_PATH to a Tinker sampler checkpoint path before running this cell.")

client = OpenAI(base_url=BASE_URL, api_key=os.environ["TINKER_API_KEY"])
response = client.completions.create(
    model=MODEL_PATH,
    prompt="Compare template A and template B. Answer same or different.",
    max_tokens=8,
    temperature=0.0,
)

print(response.choices[0].text)